In [ ]:
import hashlib
import json
from pathlib import Path
import tarfile
import time

started = time.monotonic()
input_root = Path('/kaggle/input')
working = Path('/kaggle/working')
candidate_paths = [
    Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train'),
    Path('/kaggle/input/biohub-cell-tracking-during-development/train'),
]
candidate_paths.extend(path / 'train' for path in input_root.iterdir() if path.is_dir())
candidates = [path for path in candidate_paths if path.is_dir() and len(list(path.glob('*.geff'))) == 199]
candidates = list(dict.fromkeys(candidates))
if len(candidates) != 1:
    raise RuntimeError({'eligible_train_roots': [str(path) for path in candidates]})
train_root = candidates[0]
geffs = sorted(path for path in train_root.glob('*.geff') if path.is_dir())
stems = [path.stem for path in geffs]
counts = {prefix: sum(stem.startswith(prefix + '_') for stem in stems) for prefix in ('44b6', '6bba')}
if len(geffs) != 199 or counts != {'44b6': 71, '6bba': 128}:
    raise RuntimeError({'geffs': len(geffs), 'embryos': counts})
print({'train_root': str(train_root), 'movies': len(geffs), 'embryos': counts}, flush=True)
records = []
for geff_index, geff in enumerate(geffs, start=1):
    for path in sorted(member for member in geff.rglob('*') if member.is_file()):
        relative = path.relative_to(train_root).as_posix()
        records.append({'relative_path': relative, 'bytes': path.stat().st_size, 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
    if geff_index % 25 == 0:
        print(f'hashed {geff_index}/{len(geffs)} GEFFs', flush=True)
if len(records) < 3000 or any(not row['relative_path'].endswith(('zarr.json', '/0', '/0/0')) for row in records):
    raise RuntimeError({'unexpected_member_count': len(records)})
manifest = {
    'schema_version': 1,
    'status': 'complete',
    'run_id': 'competition-train-geff-cache-v1',
    'stems': stems,
    'files': records,
    'summary': {'movies': len(stems), 'members': len(records), 'bytes': sum(row['bytes'] for row in records), 'embryos': counts},
    'competition_train_data_read': True,
    'competition_test_data_read': False,
    'public_leaderboard_used_for_selection': False,
    'submission_created': False,
    'authorized_for_submission': False,
}
manifest_path = working / 'train_geff_cache_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')
archive_path = working / 'biohub_train_geffs_v1.tar.gz'
with tarfile.open(archive_path, 'w:gz') as archive:
    for geff in geffs:
        archive.add(geff, arcname=f'train/{geff.name}', recursive=True)
    archive.add(manifest_path, arcname=manifest_path.name)
terminal = {
    'schema_version': 1,
    'status': 'completed',
    'run_id': 'competition-train-geff-cache-v1',
    'accelerator': 'cpu',
    'gpu_used': False,
    'elapsed_seconds': round(time.monotonic() - started, 3),
    'archive_bytes': archive_path.stat().st_size,
    'archive_sha256': hashlib.sha256(archive_path.read_bytes()).hexdigest(),
    'manifest_sha256': hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
    'competition_train_data_read': True,
    'competition_test_data_read': False,
    'submission_created': False,
}
(working / 'launcher_terminal.json').write_text(json.dumps(terminal, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(terminal, indent=2, sort_keys=True))
